# 4. Examining the context of words

## Concordances

Lists of frequent words can be very useful. They can help to clarify the main concerns or the themes of a text, for example. To examine *how* words are used in a text in more detail, however, it can also be useful to create a concordance. In a concordance, all the occurrences of a given search term are listed in combination with words that occur before and after this term. Such resources are sometimes referred to as *keyword in context* lists (KWIC). 

The `nltk` package contains a method named `concordance()`. To work with this method, you firstly need to create an instance of the `Text` class. This class is part of the `nltk.text` module. Such a `Text` object can be initialised using a list with all the tokens of a text. 


In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.text import Text
import os

path = os.path.join('Corpus','PrideAndPrejudice.txt')

with open( path , encoding = 'utf-8') as file:
    full_text = file.read()

tokens = word_tokenize(full_text)
novel = Text(tokens)

In the code above, the `Text` object is given the name `novel`. 

Once you have created such an object, you can use its `concordance()` method. You can supply three parameters: 

1. A search term.
2. A `width` parameter, specifying the total length of the fragment that is extracted from the text, measured in characters. With this parameter, you indicate the number of characters before and after the word whose context you want to see. 
3. A `lines` parameter, which specifies the number of results. 

Out of these parameters, only the first one is mandatory. When you leave out the last two parameters, the method will work with its default values: a fragment consisting of 70 characters and 25 lines. 

In [ ]:
novel.concordance('hope' , width = 50 , lines = 15)

In the `concordance()` method that is defined in `nltk`, the width of the context is defined using a specific number of characters. When you work with such a fixed number of characters, the search term can be shown at the same position on each line, resulting in a keyword-in-context list with a nice and orderly appearance.

The downside of this approach is that the various lines may contain incomplete words. Which you indicate that the size of the fragment must be set at 50 characters long, the code simply removes all characters before or beyond this number of characters. 

The cell below contains a definition of a method which can create a somwhat different type of concordance. In this method, named `concordance_word()`, the width of the context is specified using words rather characters. When you supply the number 5 as the value for the parameter defining the width, you will receive all occurrences of the search term, together with 5 words before and 5 words after this search term. The method demands a sting as input. This string can be the full text of a novel. 

The results are returned as a list. 

In [ ]:

import re


def concordance_word( text, regex , distance ):

    concordance = []

    segment_length = 0

    words = word_tokenize( text )
    words = remove_punctuation( words )
    i = 0
    for w in words:
        if re.search( regex , w , re.IGNORECASE ):
            match = ''
            for x in range( i - distance , ( i + distance ) + 1 ):
                if x >= 0 and x < len(words):
                    if len(words[x]) >= 0:
                        match += words[x] + ' '
            concordance.append( match )

        i += 1

    return concordance



The cell below contains an illustration of how you can use this method. 

In [ ]:
from text_mining import *

path = os.path.join('Corpus','BraveNewWorld.txt')

word = 'savage'

with open( path , encoding = 'utf-8') as file:
    full_text = file.read()
    
fragments = concordance_word( full_text , word , 10)

print( f'There are {len(fragments)} ocurrences of the word "{word}".')

number_of_results = 5

print( f'Here are the first {number_of_results} occurrences:\n\n')
for f in fragments[:number_of_results]:
    print( f'{f}')

As you can see in the definition of `concordance_word()`, the method searches for occurrences of the supplied search term as a regular expression. The second parameter of this method can also be a more complicated regular expression. 

In [ ]:
fragments = concordance_word(full_text , r'(\bhates?\b)|(\bloves?\b)' , 10)

for f in fragments[:10]:
    print( f'{f}')

## Bigrams

An n-gram is a sequence of adjacent units. These units can be words, for example. A bigram is a specific type of n-gram. It a sequence of **two** adjacent or co-occurring words. On the basis of a bigram analysis, we can examine which words are most frequently combined. 

The `bigrams()` method from `ntlk` demands a list of words as input. It returns a `generator` object, containing all possible bigrams. The output of this method can be counted by converting it to a `Counter()` object. 

In the opening paragraph of Dickens' *A Tale of Two Cities*, 'it was' and 'was the' are the most frequent bigrams. 

In [ ]:
import nltk
from nltk import word_tokenize
from nltk.collocations import *
from text_mining import *

fragment = '''
It was the best of times, it was the worst of times,
it was the age of wisdom, it was the age of foolishness,
it was the epoch of belief, it was the epoch of incredulity,
it was the season of light, it was the season of darkness, 
it was the spring of hope, it was the winter of despair
'''

words = word_tokenize(fragment.lower())
words = remove_punctuation(words)

bigrams = Counter(nltk.bigrams(words))
for bigram,count in bigrams.most_common():
    print(f'{bigram} => {count}')

`nltk` offers a similar method for the analysis of trigrams.

In [ ]:
trigrams = Counter(nltk.trigrams(words))
for trigram,count in trigrams.most_common():
    if count>1:
        print(f'{trigram} => {count}')

## Collocation analysis

Collocation analyses focus on the words that are used in the vicinity of a provided search term. It may be viewed as an extension of the principle underlying the creation of concordances. To perform a collocation analysis, we can look at the environments of a search term through a 'window' consisting of a given number of words. The words that are used in this context can obviously be counted. The aim of a collocation analysis is to identify the words that are used most frequently in the neighbourhood of a given word. 

Such collocation analyses can be carried out using the `collocation()` method that is defined below. 

In [ ]:
def collocation( text , regex , distance ):

    freq_c = dict()

    sentences = sent_tokenize( text )

    for sentence in sentences:

        words = word_tokenize( sentence )
        words = remove_punctuation(words)

        for i,w in enumerate(words):
            if re.search( regex , w , re.IGNORECASE ):
                index_regex = i 

                for x in range( i - distance , i + distance ):
                    if x >= 0 and x < len(words) and words[x].lower() != words[index_regex].lower():
                        if len(words[x]) > 0:
                            word = words[x].lower()
                            freq_c[ word ] = freq_c.get( word , 0 ) + 1
            
    return freq_c


The parameters are the same as those of the `concordance_word()` method: 

1. The text that needs to be analysed.
2. A search term, which will be treated as a regular expression.
3. A number representing the width of the context (or, ot be more precise: the number of words). 

This function returns a dictionary listing all the words found near the search term that is provided, together with the frequencies of these words. In the code below, this dictionary is converted t a `Counter()` object, to be able to sort the collocations by frequency. 

In [ ]:
nearby_words = collocation( full_text , r'marriage' , 10)
nearby_words_counted = Counter()
nearby_words_counted.update(nearby_words)

from nltk.corpus import stopwords
stopwords = stopwords.words('english')

for word,count in nearby_words_counted.most_common():
    if word not in stopwords and count>2:
        print(f'{word} => {count}')

## Cooccurrence

Once you have established that two specific words are often used in combination, you can begin to study specific combinations of words in more detail using the `cooccurrence()` method that is defined below.

In [ ]:
def cooccurrence( text , word1 , word2 , width ):
    
    relevant_sentences = []
    
    text = re.sub( '\s+' , ' ' , text )
    sentences = sent_tokenize( text )

    for s in sentences:
        if re.search( r'\b' + word1 + r'\b' , s , re.IGNORECASE ) and re.search( r'\b' + word2 + r'\b' , s , re.IGNORECASE ):

            words = word_tokenize(s)
            word1_indexes = []
            word2_indexes = []
            
            for i,w in enumerate(words):
                if re.search( r'\b' + word1 + r'\b' , w , re.IGNORECASE ):
                    word1_indexes.append(i)
                elif re.search( r'\b' + word2 + r'\b' , w , re.IGNORECASE ):
                    word2_indexes.append(i)

            if word1_indexes[0] > word2_indexes[0]:
                difference = word1_indexes[0] - word2_indexes[0]
            else:
                difference = word2_indexes[0] - word1_indexes[0]

            if difference <= width:
                relevant_sentences.append(s)
    return relevant_sentences
                       

The useage of the method is as follows:
    
* As the first parameter, you mus provide the full text that you want to analyse, as a single string.
* As the second and the third parameter, you need to mention the two words that you are interested in. 
* How close should these two words be? The fourth parameter specifies the number of words that are allowed in between the two words you focus on.  

The method `cooccurrence()` returns all the sentences containing the two words that you focus on. The distance (measured in number of words) will not be greater than the width that you specified. 

In [ ]:
sentences = cooccurrence( full_text , 'marriage' , 'happiness' , 10 )

for s in sentences:
    print( f'{s}\n')

# Exercises
    

## Exercise 4.1

Create a concordance for the word 'freedom' in the novel *1984*. You can find the full text in the 'Corpus' folder. Each fragment shoudl have a length of 50 characters. Show the context of the first 20 occurrences only. 

In [ ]:

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.text import Text
import os

path = os.path.join('Corpus','1984.txt')


with open( path , encoding = 'utf-8') as file:
    full_text = file.read()



## Exercise 4.2. 

Create a concordance for the word 'brother' in the novel **1984**. This time, work with fragments of 20 words long. Display the first 15 occurrences. 

In [ ]:
import os
path = os.path.join('Corpus','1984.txt')

with open( path , encoding = 'utf-8') as file:
    full_text = file.read()


## Exercise 4.3

From the novel *Pride and Prejudice*, select all the bigrams containing as the second word 'man'. Additionally, select all the bigrams in which the second word is woman. In other words, which tokens occur before these two words in the novel? In all cases, only show the bigrams that occur more than once.

In [ ]:
import nltk
from nltk import word_tokenize
from nltk.collocations import *
from text_mining import *

path = os.path.join('Corpus','PrideAndPrejudice.txt')
full_text = open(path,encoding='utf-8').read()


## Exercise 4.4

In *Pride and Prejudice*, which words are used most frequently in the vicinity of the word 'Darcy'? Consider fragments of 7 words long.

In [ ]:
path = os.path.join('Corpus','PrideAndPrejudice.txt')
full_text = open(path,encoding='utf-8').read()



## Exercise 4.5

Find all the sentences in *1984* containing the words 'war' and 'peace'. Make sure that, in these sentences, there are no more than 10 words in between these two terms.  

In [ ]:

import os
path = os.path.join('Corpus','1984.txt')

with open( path , encoding = 'utf-8') as file:
    full_text = file.read()
        
